## Visualising interactive maps
This notebook is a simple guide on how to visualize raster images using the **leafmap** and **localtileserver** libraries in combination with **jupyter-server-proxy**.

Using these packages locally is quite easy; however, in a JupyterHub environment, raster visualisation libraries typically need **jupyter-server-proxy** to function.

To help users configure these packages using environment variables, this notebook was created.

In [7]:
# For this notebook, we will use the following rasters:
product_path_1 = "/data/MTDA/TERRASCOPE_Sentinel1/SLC_COHERENCE/2025/02/19/S1A_S1A_Coherence_20250207T054220_20250219T054219_DSC_139_V110/S1A_S1A_Coherence_20250207T054220_20250219T054219_DSC_139_V110_VV.tif"
product_path_2 = "/data/MTDA/TERRASCOPE_Sentinel1/SLC_COHERENCE/2025/02/19/S1A_S1A_Coherence_20250207T054310_20250219T054309_DSC_139_V110/S1A_S1A_Coherence_20250207T054310_20250219T054309_DSC_139_V110_VV.tif"

import os.path
if os.path.exists(product_path_1):
	print(f"The file {product_path_1} exists.")
if os.path.exists(product_path_2):
	print(f"The file {product_path_2} exists.")

The file /data/MTDA/TERRASCOPE_Sentinel1/SLC_COHERENCE/2025/02/19/S1A_S1A_Coherence_20250207T054220_20250219T054219_DSC_139_V110/S1A_S1A_Coherence_20250207T054220_20250219T054219_DSC_139_V110_VV.tif exists.
The file /data/MTDA/TERRASCOPE_Sentinel1/SLC_COHERENCE/2025/02/19/S1A_S1A_Coherence_20250207T054310_20250219T054309_DSC_139_V110/S1A_S1A_Coherence_20250207T054310_20250219T054309_DSC_139_V110_VV.tif exists.


### Localtileserver
The localtileserver allows users to create a backend tileserver that converts local raster data (e.g. GeoTIFFs) into web map tiles (XYZ tiles) so they can be displayed interactively in a web map or notebook. We can use the tileserver in combination with the ipyleaflet library to visualise rasters. Here is an example on how to run this package.

In [8]:
from localtileserver import get_leaflet_tile_layer, TileClient
from ipyleaflet import Map, FullScreenControl
import os

# Update LOCALTILESERVER_CLIENT_PREFIX to handle the JUPYTERHUB_SERVICE_PREFIX.
if os.environ.get('JUPYTERHUB_SERVICE_PREFIX') and 'LOCALTILESERVER_CLIENT_PREFIX' not in os.environ:
    os.environ['LOCALTILESERVER_CLIENT_PREFIX'] = f"{os.environ['JUPYTERHUB_SERVICE_PREFIX']}proxy/{{port}}"

# After which, you can create a tile server from local raster file
tile_client_1 = TileClient(product_path_1)
tile_client_2 = TileClient(product_path_2)

# Create the layers using the tileservers
layer1 = get_leaflet_tile_layer(tile_client_1)
layer2 = get_leaflet_tile_layer(tile_client_2)

# Create the map and add the layers
m = Map(center=tile_client_2.center(), zoom=4.5)
m.add_layer(layer1)
m.add_layer(layer2)
m.add_control(FullScreenControl())

# Visualise
m

Map(center=[48.4472805, 6.938029500000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

### Leafmap
Similarly to localtileserver, Leafmap is a package used to create interactive maps with minimal coding. It is built on several open-source libraries such as folium and ipyleaflet, which make it easy to generate interactive visualizations. Below is a code snippet that demonstrates how to visualize a raster.

Leafmap appends an extra / to the end of the JUPYTERHUB_SERVICE_PREFIX. However, by default, JUPYTERHUB_SERVICE_PREFIX already ends with a /. This duplication can cause rendering issues, which can be resolved by **temporarily** removing the trailing / from the prefix.

In [9]:
import os
import leafmap.leafmap as leafmap

if os.environ.get('JUPYTERHUB_SERVICE_PREFIX'):
    os.environ['JUPYTERHUB_SERVICE_PREFIX'] = os.environ['JUPYTERHUB_SERVICE_PREFIX'][:-1] if os.environ['JUPYTERHUB_SERVICE_PREFIX'][-1] == "/" else os.environ['JUPYTERHUB_SERVICE_PREFIX']

m = leafmap.Map()
m.add_raster(product_path_1)
m.add_raster(product_path_2)
m

Map(center=[48.4472805, 6.938029500000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

When you are done with your render, add the '/' back at the end of the prefix.

In [10]:
if os.environ.get('JUPYTERHUB_SERVICE_PREFIX'):
    os.environ['JUPYTERHUB_SERVICE_PREFIX'] = os.environ['JUPYTERHUB_SERVICE_PREFIX'] if os.environ['JUPYTERHUB_SERVICE_PREFIX'][-1] == "/" else f"{os.environ['JUPYTERHUB_SERVICE_PREFIX']}/"